In [ ]:
import warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from gaitutils.stats import collect_trial_data
from gaitutils.trial import Trial

In [ ]:
NORMAL_DATA_FNAME = 'Z:/andrey/Verbatim/Monitaso/TD_normaldata_all.xlsx'
CATALOG_FNAME = 'Z:/andrey/Verbatim/Monitaso/Monitasodata_taulukko.xlsx'
DATA_DIR = 'Z:/andrey/Verbatim/Monitaso'

NORM_TRIAL_LEN = 101    # number of points to which each cycle is resampled

VAR_NAME_MAP = {
    'AnkleAnglesX': 'AnkleAngles (1)',
    'KneeAnglesX': 'KneeAngles (1)',
    'HipAnglesX': 'HipAngles (1)',
    'HipAnglesY': 'HipAngles (2)',
    'HipAnglesZ': 'HipAngles (3)',
    'PelvisAnglesX': 'PelvisAngles (1)',
    'PelvisAnglesY': 'PelvisAngles (2)',
    'PelvisAnglesZ': 'PelvisAngles (3)',
    'FootProgressAnglesZ': 'FootProgressAngles (3)'
}

## Read and prepare the normal data

In [ ]:
df_normal = pd.read_excel(NORMAL_DATA_FNAME, header=0, skiprows=[1, 2])
norm_data = {}

for var_name in VAR_NAME_MAP:
    vals_low = df_normal[VAR_NAME_MAP[var_name]].to_numpy()
    vals_high = df_normal[VAR_NAME_MAP[var_name] + '.1'].to_numpy()

    x_old = np.linspace(0, 1, len(vals_low))
    x_new = np.linspace(0, 1, NORM_TRIAL_LEN)

    norm_data[var_name] = np.interp(x_new, x_old, (vals_low + vals_high) / 2)

## Read the patient data

### Define some aux functions

In [ ]:
def read_cond(row, path_header, tags_header):
    if pd.isna(row[path_header]) or pd.isna(row[tags_header]):
        return None
    
    relpath = row[path_header].replace('\\', '/')

    fullpath = Path(DATA_DIR) / relpath
    assert fullpath.is_dir(), f"Directory {fullpath} does not exist"

    tags = row[tags_header].split(',')
    tags = [tag.strip() for tag in tags]
    print(f"Processing {fullpath} with tags {tags} ...")

    res = {'R': [], 'L': []}

    for c3d_file in fullpath.glob('*.c3d'):
        try:
            trial = Trial(c3d_file)

            tag_hits = ((tag in trial.eclipse_data['DESCRIPTION']) or (tag in trial.eclipse_data['NOTES']) for tag in tags)

            if any(tag_hits):
                print(f'File {c3d_file} is tagged, collecting data...')
                data, cycles = collect_trial_data(trial, analog_envelope=True, force_collect_all_cycles=True, fp_cycles_only=False)

                for side in ['R', 'L']:

                    var_mean_sqs = []
                    for var_name in VAR_NAME_MAP:
                        var_mean_sq = ((data['model'][f'{side}{var_name}']) ** 2 - (norm_data[var_name] ** 2)).mean(axis=1)
                        var_mean_sqs.append(var_mean_sq)

                    gps_sq = np.mean(np.stack(var_mean_sqs), axis=0)
                    gps = np.sqrt(gps_sq)

                    res[side].extend(gps.tolist())
        except:
            warnings.warn(f'Error reading {c3d_file} ignoring...')

    if len(res['R']) == 0 and len(res['L']) == 0:
        print(f'No gait cycles found in {fullpath} with tags {tags}')
        return None
    
    return res

In [ ]:
def read_row(row):
    if pd.isna(row['Potilaskoodi']):
        return None

    res = {}
    for path_header, tags_header in [('Pre', 'Pre tags'), ('Post1', 'Post1 tags'), ('Post2', 'Post2 tags'), ('Post5', 'Post5 tags')]:
        cond_res = read_cond(row, path_header, tags_header)
        if cond_res is not None:
            res[path_header] = cond_res

    if len(res) == 0:
        return None
    else:
        return row['Potilaskoodi'], res 



### Read the data

In [ ]:
all_gps = {}  # patient_id -> condition -> side -> list of GPS values

In [ ]:
df = pd.read_excel(CATALOG_FNAME, sheet_name='GPS', header=0)


for idx, row in df.iterrows():

    res_row = read_row(row)
    if res_row is not None:
        patient_id, cond_data = res_row
        all_gps[patient_id] = cond_data

In [ ]:
len(all_gps)

In [ ]:
for aspect in ('R', 'L'):
    for cond in ('Pre', 'Post1', 'Post2', 'Post5'):

        pat_ids = []
        avg_gps = np.zeros((len(all_gps),))
        std_gps = np.zeros((len(all_gps),))

        for i, pat_id in enumerate(all_gps):
            pat_ids.append(pat_id)
            try:
                avg_gps[i] = np.mean(all_gps[pat_id][cond][aspect])
                std_gps[i] = np.std(all_gps[pat_id][cond][aspect])
            except:
                #print(f'No \'{cond}\' data for the {aspect} aspect for {pat_id}, setting to zero')
                pass

        fig = go.Figure()

        # Add the Bar Chart trace with whiskers
        fig.add_trace(go.Bar(
            x=pat_ids,
            y=avg_gps,
            name='GPS',
            marker_color='steelblue',
            error_y=dict(
                type='data',
                array=std_gps,
                visible=True,
                color='black',
                thickness=1.5,
                width=8
            )
        ))

        # Customize the layout
        fig.update_layout(
            title=f'Condition: {cond}, aspect: {aspect}',
            xaxis_title='patient',
            yaxis_title='GPS',
            template='plotly_white',
            height=500,
            width=1000,
            yaxis_range=(0, 50)
        )

        # 5. Make the x-axis labels diagonal
        fig.update_xaxes(tickangle=-45)  # -45 degrees tilts them down-left

        # 6. Show the plot
        fig.show()